# Text Generation and Summarization with the Gemini API

In this session you call a large language model (LLM) from Python using a model hosted by Google and get text back.

By the end you should be able to:

- make a request and read everything that comes back: the text, why it stopped, and how many tokens it cost;
- steer a model with a **system instruction** and keep a **multi-turn conversation**;
- control generation with the parameters that matter: `temperature`, `top_p`, `top_k`, `max_output_tokens`, `stop_sequences`, `seed` and `thinking_level`;
- get **structured output** (JSON) instead of free prose;
- summarize documents for different lengths, audiences and purposes.

Exercises are marked **Your turn**. Every call in this notebook uses a free-tier key.

## 0. Setup

### Your API key

You need a Gemini API key from Google AI Studio (<https://aistudio.google.com/apikey>). There are two ways to give it to this notebook:

1. **Colab secret (recommended).** Click the key icon in Colab's left sidebar, add a secret named `GEMINI_API_KEY`, paste the key as its value and switch on **Notebook access**. The key then lives in your Google account, not in the notebook.
2. **Paste it when asked.** If no secret is found, the setup cell prompts for the key. The input is hidden, so the key never appears in the notebook output.

In [ ]:
%pip install -q -U google-genai

In [ ]:
import getpass
import os


def get_api_key():
    """Look for the key in a Colab secret, then the environment, then ask for it."""
    try:
        from google.colab import userdata

        for name in ("GoogleAPI", "GEMINI_API_KEY", "GOOGLE_API_KEY"):
            try:
                key = userdata.get(name)
            except Exception:
                key = None  # secret missing, or notebook access not granted
            if key:
                return key, f"Colab secret {name}"
    except ImportError:
        pass  # not running on Colab

    for name in ("GEMINI_API_KEY", "GOOGLE_API_KEY"):
        if os.environ.get(name):
            return os.environ[name], f"environment variable {name}"

    return (
        getpass.getpass("Paste your Gemini API key (it will not be shown): ").strip(),
        "prompt",
    )


api_key, source = get_api_key()
# Print the shape of the key, never the key itself. Enough to spot a truncated paste.
print(f"Key found in: {source}. Length {len(api_key)}, starts with {api_key[:4]}...")

### The client

Every interaction goes through one `Client` object. It holds your key and exposes the API in groups: `client.models` (generate text, count tokens, embed), `client.chats` (conversations), `client.files`, and others.

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=api_key)

MODEL = "gemini-3.5-flash-lite"  # fast, cheap, generous free-tier quota
EMBEDDING_MODEL = "gemini-embedding-001"

Models are retired and renamed often. If a call fails with "model not found", list what your key can use and change `MODEL`. The current catalogue is at <https://ai.google.dev/gemini-api/docs/models>.

In [ ]:
for m in client.models.list():
    if "flash" in m.name or "embedding" in m.name:
        print(
            f"{m.name:<55} input limit {m.input_token_limit:>9,}  output limit {m.output_token_limit or 0:>7,}"
        )

## 1. Your first request

The basic call is `client.models.generate_content(model=..., contents=...)`. `contents` is the prompt.

In [ ]:
response = client.models.generate_content(
    model=MODEL,
    contents="Explain in two sentences what a large language model is.",
)
print(response.text)

### What comes back

`response.text` is a shortcut. The full response object carries more, and three parts of it matter in practice:

- **`candidates[0].finish_reason`**: why the model stopped. `STOP` means it finished naturally. `MAX_TOKENS` means it was cut off. `SAFETY` means a filter blocked it.
- **`usage_metadata`**: token counts. You pay per token, separately for input and output, so this is your bill.
- **`model_version`**: the exact model that answered.

In [ ]:
print("finish reason :", response.candidates[0].finish_reason)
print("model version :", response.model_version)

usage = response.usage_metadata
print("input tokens  :", usage.prompt_token_count)
print("output tokens :", usage.candidates_token_count)
print("thinking      :", usage.thoughts_token_count)  # see section 3
print("total tokens  :", usage.total_token_count)

Run `response` on its own in a cell to see the whole object at least once. It is verbose, but every field there is something the API can tell you.

In [ ]:
response

`response.candidates` is the list of generated response options returned by the API.

`candidate_count` requests multiple options, but current models may only support one candidate.

If you need more than one candidate to be returned, run a separate request for each

### Tokens

Models read and write **tokens**, not words. A token is a word or a piece of one: in English, roughly 4 characters or three quarters of a word. Context limits, prices and rate limits are all counted in tokens.

`count_tokens` tells you the size of a prompt without generating anything. It is free.

In [ ]:
for text in [
    "Hello",
    "Hello, how are you doing today?",
    "Supercalifragilisticexpialidocious",
    "Hola, ¿cómo estás hoy?",
    "您好，今天过得怎么样",
]:
    n = client.models.count_tokens(model=MODEL, contents=text).total_tokens
    print(f"{n:>3} tokens  {len(text):>3} characters  {text}")

Rare words split into several tokens, and languages other than English often need more tokens for the same meaning. The same message can cost different amounts in different languages.

## 2. System instructions

A **system instruction** sets the model's role, tone and rules for the whole request. It is kept separate from the user's message: the user prompt says *what* to do, the system instruction says *how to behave* while doing it.

All request options, including the system instruction, go in a `GenerateContentConfig` passed as `config`.

In [ ]:
question = "Should I learn Python or Excel first?"

for persona in [
    "You are a helpful assistant.",
    "You are a senior data scientist. Answer in at most three bullet points.",
    "You are a sceptical finance director. Be blunt and brief.",
    "You answer only in Spanish, in one sentence.",
]:
    response = client.models.generate_content(
        model=MODEL,
        contents=question,
        config=types.GenerateContentConfig(system_instruction=persona),
    )
    print(f"### {persona}\n{response.text}\n")

A system instruction is not a guarantee. Models usually follow it, but a determined user can override it. Never rely on a system instruction alone to keep data secret or to enforce a business rule.

## 3. Generation parameters

An LLM generates text one token at a time. At each step it computes a probability for every token in its vocabulary, then **samples** one. The parameters below control that sampling and when it stops.

| Parameter | What it does | Typical use |
|---|---|---|
| `temperature` | Flattens (high) or sharpens (low) the probability distribution. Range 0 to 2. | Low for extraction and classification; default for open writing |
| `top_p` | Samples only from the smallest set of tokens whose probabilities add up to `p`. | Cut off the long tail of unlikely tokens |
| `top_k` | Samples only from the `k` most likely tokens. | Same idea, fixed count |
| `max_output_tokens` | Hard cap on response length. | Cost control |
| `stop_sequences` | Stops generation when one of these strings appears. | Cut output at a known marker |
| `seed` | Fixes the random draws, so repeated calls tend to match. | Reproducible experiments |
| `thinking_level` | How much the model reasons internally before answering. | Higher for hard problems, lower for speed |

A note on current models: Google recommends leaving `temperature` at its default of 1.0 for Gemini 3 models, as very low values can cause repetitive output on complex tasks. We change it here to see what it does.

### Temperature

Same prompt, five calls each at three temperatures. Count how many different answers come back.

In [ ]:
prompt = "Invent a name for a coffee shop run by robots. Reply with the name only."

for temperature in [0.0, 1.0, 2.0]:
    names = []
    for _ in range(5):
        response = client.models.generate_content(
            model=MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(temperature=temperature),
        )
        names.append(response.text.strip())
    print(f"temperature {temperature}: {len(set(names))} distinct -> {names}")

At temperature 0 the model almost always takes the most likely token, so answers repeat. Higher temperatures give more variety and, past a point, less coherence. Even at 0, hosted models are not perfectly deterministic: the serving infrastructure introduces small numerical differences.

### `top_p` and `top_k`

Both remove unlikely tokens before sampling. With `top_k=1` only the single most likely token survives, so the output behaves like temperature 0.

In [ ]:
for label, config in [
    ("top_k=1", types.GenerateContentConfig(top_k=1)),
    ("top_p=0.1", types.GenerateContentConfig(top_p=0.1)),
    ("top_p=1.0", types.GenerateContentConfig(top_p=1.0)),
]:
    names = [
        client.models.generate_content(
            model=MODEL, contents=prompt, config=config
        ).text.strip()
        for _ in range(4)
    ]
    print(f"{label:<10} {len(set(names))} distinct -> {names}")

In practice, tune `temperature` and leave `top_p` and `top_k` alone. Changing several at once makes it hard to tell which one had the effect.

### `max_output_tokens` and `finish_reason`

The cap is enforced mid-sentence. Always check `finish_reason`: a truncated answer looks like a normal one if you only read `.text`.

Thinking tokens count against the cap too, so here thinking is set to the minimum to leave the budget for the answer.

In [ ]:
response = client.models.generate_content(
    model=MODEL,
    contents="Describe the history of the Internet.",
    config=types.GenerateContentConfig(
        max_output_tokens=20,
        thinking_config=types.ThinkingConfig(thinking_level="minimal"),
    ),
)
print(response.text)
print("\nfinish reason:", response.candidates[0].finish_reason)
print("output tokens:", response.usage_metadata.candidates_token_count)

### `stop_sequences`

Generation stops as soon as the model writes one of these strings. The stop string itself is not included.

In [ ]:
response = client.models.generate_content(
    model=MODEL,
    contents="List ten European capitals, numbered 1. to 10., one per line.",
    config=types.GenerateContentConfig(stop_sequences=["4."]),
)
print(response.text)
print("finish reason:", response.candidates[0].finish_reason)

### `seed`

A fixed seed makes repeated calls with the same prompt and parameters return the same output most of the time. It is best effort, not a guarantee.

In [ ]:
creative = "Write a one-line slogan for an umbrella brand."

for seed in [42, 42, 42, 7]:
    response = client.models.generate_content(
        model=MODEL,
        contents=creative,
        config=types.GenerateContentConfig(temperature=1.0, seed=seed),
    )
    print(f"seed {seed:>2}: {response.text.strip()}")

### `thinking_level`

Recent Gemini models can **think** before answering: they generate hidden reasoning tokens, then the answer. More thinking helps on multi-step problems, and costs time and tokens. You pay for thinking tokens as output tokens even though you do not see them.

`thinking_level` takes `"minimal"`, `"low"`, `"medium"` or `"high"`. The puzzle below has a tempting wrong answer, try figuring out yourself first.

In [ ]:
import time

puzzle = (
    "A bat and a ball cost 20.50 euros in total. The bat costs 10.00 euro more than "
    "the ball. How much does the ball cost? Give the final answer in one line."
)

for level in ["minimal", "high"]:
    started = time.perf_counter()
    response = client.models.generate_content(
        model=MODEL,
        contents=puzzle,
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(thinking_level=level)
        ),
    )
    seconds = time.perf_counter() - started
    usage = response.usage_metadata
    print(
        f"### thinking_level={level}: {seconds:.1f}s, "
        f"{usage.thoughts_token_count or 0} thinking tokens, "
        f"{usage.candidates_token_count} answer tokens"
    )
    print(response.text.strip(), "\n")

### 3.1 Your turn

You are building a feature that tags incoming customer emails as `billing`, `technical`, or `other`. The same email must always get the same tag.

1. Write a system instruction and a `GenerateContentConfig` for this task. Choose the parameters deliberately and add a comment per parameter saying why.
2. Run the three emails below five times each. Did every email get the same tag every time?
3. Now set `temperature=2.0` and repeat. What changed?

In [ ]:
import time
from collections import Counter
from typing import Literal

from google.genai import errors
from pydantic import BaseModel

emails = [
    "I was charged twice for my subscription this month.",
    "The app crashes every time I open the settings page.",
    "Do you have an office in Lisbon?",
    # My own: deliberately ambiguous (payment details + login problem)
    "I can't log in to update my credit card details before my renewal date.",
]

# 1) System instruction and config
TAGGER_SYSTEM = """You classify customer support emails into exactly one category.
- billing: charges, payments, invoices, refunds, prices, subscriptions, payment methods.
- technical: bugs, crashes, errors, login or access problems, performance, how the product works.
- other: anything else (company info, offices, partnerships, general questions).
If an email touches several categories, choose the one the customer needs solved first
to be able to continue. Reply with the category only."""


class Tag(BaseModel):
    tag: Literal["billing", "technical", "other"]


def tagger_config(temperature):
    return types.GenerateContentConfig(
        system_instruction=TAGGER_SYSTEM,
        # temperature: 0 -> always take the most likely token; a classifier must be repeatable, not creative
        temperature=temperature,
        # seed: fixes the random draws, an extra (best-effort) layer of reproducibility
        seed=42,
        # response_mime_type + response_json_schema: the answer can only be one of the three labels,
        # so no "Billing.", "This is billing" or invented categories to parse
        response_mime_type="application/json",
        response_json_schema=Tag.model_json_schema(),
        # max_output_tokens: the answer is a few tokens; the cap limits cost if something goes wrong
        max_output_tokens=50,
        # thinking_level: an easy task, minimal thinking keeps it fast and cheap
        thinking_config=types.ThinkingConfig(thinking_level="minimal"),
        # top_p / top_k: left at default, as recommended; we tune only temperature
    )


def classify(email, config, retries=5):
    """Tag one email. Waits and retries if we hit the free-tier rate limit."""
    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model=MODEL, contents=email, config=config
            )
            return Tag.model_validate_json(response.text).tag
        except errors.APIError as e:
            if e.code == 429 and attempt < retries - 1:
                time.sleep(15 * (attempt + 1))
            else:
                return f"error:{e.code}"
        except Exception:
            return "invalid"  # the output did not match the schema (e.g. cut off)


def run(temperature, n=5):
    config = tagger_config(temperature)
    print(f"### temperature={temperature}")
    results = {}
    for email in emails:
        tags = [classify(email, config) for _ in range(n)]
        results[email] = tags
        consistent = "OK " if len(set(tags)) == 1 else "!! "
        print(f"{consistent}{len(set(tags))} distinct {dict(Counter(tags))}  <- {email}")
    print()
    return results


# 2) Each email five times at temperature 0
results_t0 = run(0.0)

# 3) Same thing at temperature 2.0
results_t2 = run(2.0)

**Answers (3.1)**

1. **Parameter choices.** The task is classification, so we want repeatability, not creativity: `temperature=0` and a fixed `seed`. The output is constrained with a JSON schema whose only field is an enum (`billing` / `technical` / `other`), so the program never has to parse free text and the model cannot invent a fourth category. `max_output_tokens` is small because the answer is a single word, and `thinking_level="minimal"` because the task is easy and we want speed and low cost. `top_p` and `top_k` stay at their defaults so that only one parameter is changing the behaviour.

2. **Temperature 0.** Every email should get the same tag in all five runs (`1 distinct` in every row): the double charge is `billing`, the crash is `technical`, the Lisbon office is `other`. My ambiguous email (login problem to update a credit card) is the one most at risk of flipping between `billing` and `technical`; the rule in the system instruction ("the one the customer needs solved first") pushes it consistently towards `technical`. Even at temperature 0 the hosted model is not perfectly deterministic, so in production you would still log the tags and monitor them.

3. **Temperature 2.0.** The clear-cut emails usually still get the same tag, because the right label has a very high probability and a flatter distribution does not change the winner much. The difference shows up on the ambiguous email, where the probabilities of `billing` and `technical` are closer: at temperature 2.0 the tag can change from one run to the next. The schema still prevents invalid labels, but it cannot prevent the *wrong valid* label. Conclusion: for a classifier that must be consistent, keep temperature low; high temperature only adds randomness without any benefit for this task.

*(Check the printed output above: if a row shows `!!` at temperature 0, note which email it was — it is a sign that the email is genuinely ambiguous and the system instruction needs a clearer rule for it.)*

## 4. Conversations

The API is **stateless**: the model remembers nothing between calls. A chatbot's memory is an illusion created by sending the whole conversation again with every new message.

A conversation is a list of turns, each with a **role**: `user` for the person, `model` for the assistant's earlier replies.

In [ ]:
response = client.models.generate_content(
    model=MODEL,
    contents="My name is Ana and I work in logistics.",
)
print("Turn 1:", response.text, "\n")

response = client.models.generate_content(model=MODEL, contents="What is my name?")
print("Turn 2 without history:", response.text)

The second call knows nothing about the first. Now send the history explicitly.

In [ ]:
history = [
    types.Content(
        role="user", parts=[types.Part(text="My name is Ana and I work in logistics.")]
    ),
    types.Content(
        role="model", parts=[types.Part(text="Nice to meet you, Ana! How can I help?")]
    ),
    types.Content(
        role="user",
        parts=[
            types.Part(text="What is my name, and suggest one AI use case for my job.")
        ],
    ),
]

response = client.models.generate_content(model=MODEL, contents=history)
print(response.text)

Because you control the history, you can also write the `model` turns yourself. This is how you show the model examples of the behaviour you want (**few-shot prompting**), or resume a conversation from a database.

### The chat helper

`client.chats` keeps the history for you. Underneath, it does exactly what the cell above did: append each turn and resend everything.

In [ ]:
chat = client.chats.create(
    model=MODEL,
    config=types.GenerateContentConfig(
        system_instruction="You are a concise travel assistant. Answer in at most three sentences."
    ),
)

for message in [
    "I have a long weekend in October. I like hiking and good food.",
    "I'd rather stay in Spain.",
    "What should I pack?",
]:
    reply = chat.send_message(message)
    print(f"USER : {message}\nMODEL: {reply.text}\n")

In [ ]:
chat.get_history()

In [ ]:
for turn in chat.get_history():
    print(f"{turn.role:>5}: {turn.parts[0].text[:80]}")

print("\ninput tokens on the last call:", reply.usage_metadata.prompt_token_count)

Every turn resends all previous turns, so input tokens grow with the conversation. Long conversations get slower and more expensive, and eventually hit the model's context limit. Production chatbots trim or summarize old turns.

### Streaming

By default you wait for the whole answer. `generate_content_stream` returns it in chunks as they are generated, which is why chat interfaces show text appearing word by word. Total time is similar; the time to the first word is much shorter.

In [ ]:
for chunk in client.models.generate_content_stream(
    model=MODEL,
    contents="Write a short paragraph on why streaming improves the user experience of chatbots.",
):
    print(chunk.text, end="", flush=True)

## 5. Structured output

Prose is fine for people. Programs need fields. With `response_mime_type="application/json"` and a schema, the model returns JSON that matches the schema, so you can parse it reliably.

The easiest way to write a schema is a Pydantic class. The `description` of each field is sent to the model and acts as a per-field instruction.

In [ ]:
import json

from pydantic import BaseModel, Field


class Contact(BaseModel):
    name: str = Field(description="Full name of the person")
    company: str = Field(
        description="Company name, or an empty string if not mentioned"
    )
    role: str = Field(description="Job title, or an empty string if not mentioned")
    wants_meeting: bool = Field(description="True if they ask for a meeting or call")


email = (
    "Hi, this is Laura Méndez, procurement lead at Nortia Logistics. We saw your demo "
    "at the trade fair and would like to set up a 30-minute call next week."
)

response = client.models.generate_content(
    model=MODEL,
    contents=f"Extract the contact details from this email:\n\n{email}",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_json_schema=Contact.model_json_schema(),
    ),
)

print(response.text)
contact = Contact.model_validate_json(response.text)  # a Python object, not a string
print("\nwants a meeting:", contact.wants_meeting)

## 6. Summarization

Summarization is one of the most common business uses of LLMs: reports, meeting notes, support tickets, reviews, contracts. A good summary depends on three choices that you make in the prompt:

- **length**: one line, a paragraph, a page;
- **audience**: an executive, a specialist, a customer;
- **purpose**: the general gist, or only the risks, the decisions, the numbers.

The text below is the introduction of the Wikipedia article [*Hundred Years' War*](https://en.wikipedia.org/wiki/Hundred_Years%27_War), licensed [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).

In [ ]:
ARTICLE = """
The Hundred Years' War (French: Guerre de Cent Ans; 1337-1453) was a series of conflicts between the kingdoms of England and France and a civil war in France during the late Middle Ages. It emerged from feudal disputes over the Duchy of Aquitaine and was triggered by a claim to the French throne made by Edward III of England. The war grew into a broader military, economic, and political struggle involving different factions from across Western Europe, fuelled by emerging nationalism on both sides. The periodisation of the war typically charts it as taking place over 116 years. However, it was an intermittent conflict which was frequently interrupted by external factors, such as the Black Death, and several years of truces.

The Hundred Years' War was a significant conflict in the Middle Ages. During the war, five generations of kings from two rival dynasties fought for the throne of France, then the wealthiest and most populous kingdom in Western Europe. The war had a lasting effect on European history: both sides produced innovations in military technology and tactics, including professional standing armies and artillery, that permanently changed European warfare. Chivalry reached its height during the conflict and subsequently declined. Stronger national identities took root in both kingdoms, which became more centralized and gradually emerged as global powers.

The term "Hundred Years' War" was adopted by later historians and included dynastically related conflicts in Spain and Belgium, resulting in the longest military conflict in European history. The war is commonly divided into three phases, separated by truces: the Edwardian War (1337-1360), the Caroline War (1369-1389), and the Lancastrian War (1415-1453). Each side drew many allies into the conflict, with English forces initially prevailing; however, the French forces under the House of Valois ultimately retained control over the Kingdom of France. The French and English monarchies thereafter remained separate, despite the monarchs of England and Great Britain styling themselves as sovereigns of France until 1802.""".strip()

print(client.models.count_tokens(model=MODEL, contents=ARTICLE).total_tokens, "tokens")

A small helper so the rest of the section is about prompts, not plumbing.

In [ ]:
def ask(prompt, system=None, **params):
    """One call, returns the text. Extra keyword arguments go into the config."""
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(system_instruction=system, **params),
    )
    return response.text

### Length

In [ ]:
for instruction in [
    "Summarize this article in one sentence.",
    "Summarize this article in three bullet points.",
    "Summarize this article in about 100 words.",
]:
    summary_text = ask(instruction + "\n\n" + ARTICLE)
    print(f"### {instruction}\n{summary_text}\n")

### Audience and purpose

Same source, different readers. Note how the selection of facts changes, not just the wording.

In [ ]:
for system in [
    "You brief a busy executive with no history background. Explain in max 80 words why this war still matters today.",
    "You write for 10-year-olds. Max 60 words, simple words, no dates except the start and end of the war.",
    "You are a military historian. List only the changes in warfare and politics that the text attributes to the war, as bullet points.",
]:
    print(f"### {system}\n{ask(ARTICLE, system=system)}\n")

### Structured summaries

When a summary feeds a dashboard or a database, ask for fields instead of prose.

In [ ]:
class HistorySummary(BaseModel):
    headline: str = Field(description="A neutral headline, max 12 words")
    period: str = Field(description="Start and end years of the conflict")
    sides: list[str] = Field(description="The main opposing parties")
    phases: list[str] = Field(description="Each phase of the war with its dates")
    key_numbers: list[str] = Field(
        description="Each important figure or year with what it refers to"
    )
    lasting_effects: list[str] = Field(
        description="Long-term consequences mentioned in the text"
    )


response = client.models.generate_content(
    model=MODEL,
    contents=f"Summarize this article:\n\n{ARTICLE}",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_json_schema=HistorySummary.model_json_schema(),
    ),
)
summary = HistorySummary.model_validate_json(response.text)
print(json.dumps(summary.model_dump(), indent=2, ensure_ascii=False))

### Faithfulness

A summary is only useful if every claim in it is in the source. LLMs sometimes add facts that sound right but are not there (**hallucinations**), and summaries are especially prone to merging or rounding numbers.

Check the `key_numbers` above against the article. Then see what happens when the prompt asks for something the text does not contain.

In [ ]:
print(
    ask(f"Summarize this article and say how many people died in the war.\n\n{ARTICLE}")
)

The text gives no casualty figures. Any number in the answer comes from the model's training data or is invented, and nothing in the answer tells you which. For a summary, both are a problem: the reader assumes every claim comes from the source. A safer prompt says explicitly: use only the text, and say so if the text does not answer the question.

In [ ]:
GROUNDED = (
    "Answer using only the article provided. If the article does not contain the "
    "answer, say 'The article does not say.' Do not add outside knowledge."
)
print(ask(f"How many people died in the war?\n\n{ARTICLE}", system=GROUNDED))

### 6.1 Your turn

You manage a coworking space and receive the reviews below every week.

1. Define a Pydantic schema for a weekly review digest: overall sentiment (`positive`, `mixed` or `negative`), the top complaints, the top praises, and one recommended action. Generate the digest with structured output.
2. Check it for faithfulness: is every complaint and praise actually in the reviews? Then ask the model to do the check: give it the reviews and the digest, and ask it to flag any point not supported by a specific review.

In [ ]:
REVIEWS = [
    "Great location and the coffee is excellent. Wi-Fi dropped twice during my calls though.",
    "Wi-Fi is unreliable in the afternoons. Staff are friendly but couldn't fix it.",
    "Love the community events. Meeting rooms are always booked, impossible to get one.",
    "Clean, bright, quiet. Exactly what I need. A bit pricey for what you get.",
    "Third week in a row the Wi-Fi died at 4pm. Considering moving elsewhere.",
    "The new phone booths are a great addition. Coffee machine broken on Friday.",
    "Meeting room booking system is a mess, double bookings twice this week.",
    "Friendly staff, nice terrace. Price went up 15% with no improvement.",
]

# TODO: 1) a digest schema and a structured-output call
# TODO: 2) ask the model to flag unsupported points in the digest
raise NotImplementedError